# Weekly Fantasy Football Rankings vs. the Experts
CS 363M final project. Hassaan Ali, Shyamal Gupta, Amin Momin.

## 1. Introduction
**Main task:** predict each QB/RB/WR/TE's PPR fantasy points for the coming week using only pre-game info, rank players by those predictions, and compare the rankings to the FantasyPros expert consensus (ECR).

**Extensions:** (1) tiers: does a direct tier classifier beat tiers cut from our predicted points? (2) usage-profile clustering.

**Data:** public NFL data loaded with `nflreadpy`. 2019 is used only for warm-up lags, 2020-2024 for training and cross-validation, and 2025 is the held-out test set, used once at the end with every setting frozen. ECR is never a model feature.

**Recency:** next week depends most on a player's current role, team, and health, so recent games get more weight. How much more is tuned with cross-validation: recency-weighted features, training weights that fade with a game's age, and a 2025 test that refits each week on all games before it.

## 2. Setup and config

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import nflreadpy as nfl

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

In [2]:
# Config: every tunable setting lives here.
SEED = 363
POSITIONS = ["QB", "RB", "WR", "TE"]
WARMUP_SEASON = 2019             # lags only, never a target row
TRAIN_SEASONS = list(range(2020, 2025))
TEST_SEASON = 2025               # used once, in the final test section
ALL_SEASONS = [WARMUP_SEASON] + TRAIN_SEASONS + [TEST_SEASON]
TARGET = "fantasy_points_ppr"

# Player pool: who we model in week t, decided from games before t only.
POOL_LOOKBACK_GAMES = 3          # look at the player's last N games played
POOL_MIN_SNAP_SHARE = 0.40       # mean offensive snap share over those games
ECR_PAGES = {"weekly-qb": "QB", "weekly-rb": "RB", "weekly-wr": "WR", "weekly-te": "TE"}  # PPR weekly pages

# Features and evaluation
HALF_LIFE_CANDIDATES = [3, 5, 8, 12, 16]  # games; recency weight halves every N games (picked by CV)
PLAIN_WINDOW_GAMES = 8            # the equal-weight version recency has to beat
LUCK_HALF_LIFE_GAMES = 16         # luck evens out slowly, so it gets a longer memory
CV_VALID_SEASONS = [2022, 2023, 2024]  # expanding window: train on every season before each
TOP_N = 12                        # "did our top 12 finish top 12" per position-week

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
for d in (RAW_DIR, PROCESSED_DIR):
    d.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)

## 3. Load and clean

Goal: one clean table with one row per player per week. Steps:
1. Download each source once and cache it in `data/raw/`.
2. Standardize team codes and map every source to one player ID (`gsis_id`).
3. Build every regular-season game a QB/RB/WR/TE played (`player_games`), the history all later lags use.
4. Pick the player pool for each week using only earlier games, then drop players who did not play.
5. Attach Vegas lines and expected points, and match each expert (ECR) snapshot to the right week.
6. Run an anomaly-detection pass to separate data errors from real big games.

In [3]:
def load_cached(name, loader):
    """Load a source from data/raw/ if cached, otherwise download it with nflreadpy and cache it."""
    path = RAW_DIR / f"{name}.parquet"
    if not path.exists():
        loader().write_parquet(path)
    return pd.read_parquet(path)


raw = {
    "stats": load_cached("player_stats", lambda: nfl.load_player_stats(ALL_SEASONS)),
    "snaps": load_cached("snap_counts", lambda: nfl.load_snap_counts(ALL_SEASONS)),
    "rosters": load_cached("rosters_weekly", lambda: nfl.load_rosters_weekly(ALL_SEASONS)),
    "schedules": load_cached("schedules", lambda: nfl.load_schedules(ALL_SEASONS)),
    "opportunity": load_cached("ff_opportunity", lambda: nfl.load_ff_opportunity(ALL_SEASONS)),
    "ids": load_cached("ff_playerids", nfl.load_ff_playerids),
    "ecr": load_cached("ff_rankings", lambda: nfl.load_ff_rankings("all")),
}
pd.DataFrame({name: df.shape for name, df in raw.items()}, index=["rows", "cols"]).T

,rows,cols
stats,129812,150
snaps,181478,16
rosters,327704,36
schedules,1960,46
opportunity,41696,159
ids,12541,35
ecr,1885069,24


### 3.1 Team codes and player IDs
The nflverse sources agree on team codes except `OAK` (2019). FantasyPros uses older codes (`JAC`, `LAR`, `SD`, `STL`).
Every source is mapped to the NFL `gsis_id`. Snap counts use Pro Football Reference IDs and ECR uses FantasyPros IDs,
so we translate them with the weekly rosters and the ffverse ID crosswalk.

In [4]:
TEAM_FIXES = {"OAK": "LV", "JAC": "JAX", "LAR": "LA", "STL": "LA", "SD": "LAC", "FA": None, "": None}


def fix_team(codes):
    return codes.replace(TEAM_FIXES)


def id_map(pairs, from_col):
    """One gsis_id per foreign ID; drop foreign IDs that point to more than one player."""
    pairs = pairs.dropna().drop_duplicates()
    pairs = pairs[~pairs[from_col].duplicated(keep=False)]
    return pairs.set_index(from_col)["gsis_id"]


ids = raw["ids"].assign(fantasypros_id=lambda d: d["fantasypros_id"].astype("Int64").astype("string"))  # stored as float
rosters = raw["rosters"]
pfr_to_gsis = id_map(pd.concat([rosters[["pfr_id", "gsis_id"]], ids[["pfr_id", "gsis_id"]]]), "pfr_id")
fp_to_gsis = id_map(ids[["fantasypros_id", "gsis_id"]], "fantasypros_id")
print(f"PFR IDs mapped: {len(pfr_to_gsis):,}   FantasyPros IDs mapped: {len(fp_to_gsis):,}")

PFR IDs mapped: 7,853   FantasyPros IDs mapped: 4,782


### 3.2 Every game each QB/RB/WR/TE played
A player "played" a game if the player took an offensive snap or recorded any stat. Players who played but had no stat line
score 0. This table is the history that the player pool and every lag feature are built from.

In [5]:
def regular_season(df, col="season_type"):
    return df[df[col] == "REG"]


def build_snaps(snaps):
    snaps = regular_season(snaps, "game_type").copy()
    snaps["gsis_id"] = snaps["pfr_player_id"].map(pfr_to_gsis)
    snaps["team"] = fix_team(snaps["team"])
    keep = ["gsis_id", "season", "week", "team", "offense_snaps", "offense_pct"]
    return snaps.dropna(subset=["gsis_id"])[keep].sort_values("offense_snaps").drop_duplicates(["gsis_id", "season", "week"], keep="last")


def build_player_games(stats, snaps, rosters):
    stats = regular_season(stats).rename(columns={"player_id": "gsis_id"})
    stats["team"] = fix_team(stats["team"])
    key = ["gsis_id", "season", "week"]
    games = stats.merge(snaps, on=key, how="outer", suffixes=("", "_snap"))
    games["team"] = games["team"].fillna(games["team_snap"])
    # Position from that week's roster, falling back to the stats feed.
    pos = regular_season(rosters, "game_type")[key + ["position"]].drop_duplicates(key)
    games = games.merge(pos, on=key, how="left", suffixes=("_stats", ""))
    games["position"] = games["position"].fillna(games["position_stats"])
    played = games["offense_snaps"].fillna(0).gt(0) | games["fantasy_points_ppr"].notna()
    games = games[played & games["position"].isin(POSITIONS)].copy()
    games[TARGET] = games[TARGET].fillna(0.0)
    return games.drop(columns=["team_snap", "position_stats"])


snaps = build_snaps(raw["snaps"])
player_games = build_player_games(raw["stats"], snaps, rosters)
print(f"{len(player_games):,} player-games;  with a snap record: {player_games['offense_snaps'].notna().mean():.1%};"
      f"  with a stat line: {player_games['player_name'].notna().mean():.1%}")
player_games.groupby(["season", "position"]).size().unstack()

46,945 player-games;  with a snap record: 99.8%;  with a stat line: 87.4%


position,QB,RB,TE,WR
season,,,,
2019,614,1707,1407,2493
2020,649,1757,1504,2590
2021,689,1785,1625,2777
2022,654,1812,1645,2704
2023,698,1703,1612,2811
2024,682,1721,1678,2751
2025,687,1759,1666,2765


**What we see:** 46,945 played games from 2019 to 2025, about 6,700 per season. 99.8% have a snap record.
12.6% have no stat line (mostly blocking snaps), and those score 0.

### 3.3 Game context and expected points
Each game gets its Vegas lines (from the schedule) and the player's expected PPR points (ffopportunity).
ffopportunity only lists players who had a pass, carry, or target, so a missing row means zero expected points.

In [6]:
def build_team_games(schedules):
    """One row per team per regular-season game, with the Vegas lines from that team's side."""
    s = regular_season(schedules, "game_type").copy()
    s["kickoff"] = pd.to_datetime(s["gameday"] + " " + s["gametime"].fillna("13:00"))
    common = ["game_id", "season", "week", "kickoff", "total_line"]
    home = s[common + ["home_team", "away_team", "spread_line"]].rename(columns={"home_team": "team", "away_team": "opponent"})
    away = s[common + ["away_team", "home_team", "spread_line"]].rename(columns={"away_team": "team", "home_team": "opponent"})
    home["is_home"], away["is_home"] = 1, 0
    away["spread_line"] = -away["spread_line"]   # nflverse spread_line is points the home team is favored by
    games = pd.concat([home, away]).rename(columns={"spread_line": "team_favored_by", "game_id": "sched_game_id"})
    for col in ("team", "opponent"):
        games[col] = fix_team(games[col])
    return games


def build_expected_points(opportunity):
    o = opportunity.rename(columns={"player_id": "gsis_id"}).dropna(subset=["gsis_id"])
    o = o.astype({"season": int, "week": int})
    return o[["gsis_id", "season", "week", "total_fantasy_points", "total_fantasy_points_exp"]].rename(
        columns={"total_fantasy_points": "pbp_points", "total_fantasy_points_exp": "expected_points"})


team_games = build_team_games(raw["schedules"])
expected = build_expected_points(raw["opportunity"])
player_games = (player_games
                .merge(team_games, on=["season", "week", "team"], how="left", validate="many_to_one")
                .merge(expected, on=["gsis_id", "season", "week"], how="left", validate="one_to_one"))
player_games["has_expected_points"] = player_games["expected_points"].notna()
player_games["expected_points"] = player_games["expected_points"].fillna(0.0)
print(f"Matched to a scheduled game: {player_games['sched_game_id'].notna().mean():.2%};"
      f"  has an ffopportunity row: {player_games['has_expected_points'].mean():.1%}")

Matched to a scheduled game: 100.00%;  has an ffopportunity row: 78.6%


### 3.4 Anomaly detection: data errors vs. real big games
- **Rule checks** find rows that cannot be right: reported PPR points that disagree with the box score,
  more receptions than targets, or touches with zero offensive snaps. These are data errors and are dropped.
- **Cross-source check:** ffopportunity recomputes actual points from play-by-play, an independent source.
  We compare it to the official box score after removing return TDs (not scored by ffopportunity) and whole
  multiples of 2 points (fumble and 2-point conversion scoring differences).
- **Isolation Forest** flags rows that are unusual given points, snap share, and opportunities.
  A row flagged only here is a real big (or very bad) game and is kept.

In [7]:
def rebuild_ppr(g):
    f = g.select_dtypes("number").fillna(0)
    return (0.04 * f["passing_yards"] + 4 * f["passing_tds"] - 2 * f["passing_interceptions"]
            + 0.1 * (f["rushing_yards"] + f["receiving_yards"]) + 6 * (f["rushing_tds"] + f["receiving_tds"])
            + f["receptions"] + 2 * (f["passing_2pt_conversions"] + f["rushing_2pt_conversions"] + f["receiving_2pt_conversions"])
            - 2 * (f["sack_fumbles_lost"] + f["rushing_fumbles_lost"] + f["receiving_fumbles_lost"])
            + 6 * f["special_teams_tds"])


def opportunities(g):
    return g["attempts"].fillna(0) + g["carries"].fillna(0) + g["targets"].fillna(0)


def rule_flags(g):
    return pd.DataFrame({
        "points_mismatch": (rebuild_ppr(g) - g[TARGET]).abs() > 0.5,
        "receptions_gt_targets": g["receptions"].fillna(0) > g["targets"].fillna(0),
        "touches_no_snaps": g["offense_snaps"].eq(0) & opportunities(g).gt(0),
    }, index=g.index)


def cross_source_gap(g):
    """Box-score points minus play-by-play points, after known scoring-rule differences."""
    gap = g[TARGET] - 6 * g["special_teams_tds"].fillna(0) - g["pbp_points"]
    scoring_rule = (gap / 2 - (gap / 2).round()).abs() < 0.01
    return gap.where(~scoring_rule, 0.0)


def isolation_flags(g):
    from sklearn.ensemble import IsolationForest
    X = pd.DataFrame({"points": g[TARGET], "snap_pct": g["offense_pct"].fillna(0), "opportunities": opportunities(g)})
    model = IsolationForest(contamination=0.01, random_state=SEED).fit(X)
    return pd.Series(model.predict(X) == -1, index=g.index)


rules = rule_flags(player_games)
player_games["data_error"] = rules.any(axis=1)
player_games["isolation_outlier"] = isolation_flags(player_games)
gap = cross_source_gap(player_games).dropna()

display(rules.sum().rename("rows flagged").to_frame())
print(f"Cross-source: {(gap.abs() <= 1).mean():.2%} of {len(gap):,} games agree within 1 point; "
      f"{(gap.abs() > 1).sum()} differ by more than 1, {(gap.abs() > 5).sum()} by more than 5.")
display(pd.crosstab(player_games["isolation_outlier"], player_games["data_error"],
                    rownames=["isolation outlier"], colnames=["data error"]))
show = ["player_display_name", "position", "season", "week", TARGET, "offense_pct", "carries", "targets"]
display(player_games[player_games["isolation_outlier"]].nlargest(5, TARGET)[show])
display(player_games[player_games["data_error"]][show])

,rows flagged
points_mismatch,0
receptions_gt_targets,0
touches_no_snaps,12


Cross-source: 99.85% of 36,876 games agree within 1 point; 56 differ by more than 1, 3 by more than 5.


data error,False,True
isolation outlier,,
False,46463,12
True,470,0


,player_display_name,position,season,week,fantasy_points_ppr,offense_pct,carries,targets
10675,Tyreek Hill,WR,2020,12,57.9,0.86,0.0,15.0
16008,Alvin Kamara,RB,2020,16,56.2,0.60,22.0,5.0
35171,Ja'Marr Chase,WR,2021,17,55.6,0.91,0.0,12.0
35210,Ja'Marr Chase,WR,2024,10,55.4,0.94,0.0,17.0
43719,Jahmyr Gibbs,RB,2025,12,55.4,0.74,15.0,12.0


,player_display_name,position,season,week,fantasy_points_ppr,offense_pct,carries,targets
3745,Giovani Bernard,RB,2022,15,0.00,0.0,1.0,0.0
5247,AJ McCarron,QB,2020,17,0.80,0.0,0.0,0.0
11358,Mike Thomas,WR,2019,15,1.50,0.0,0.0,1.0
19572,Logan Woodside,QB,2020,11,0.28,0.0,0.0,0.0
20063,Godwin Igwebuike,RB,2021,18,0.00,0.0,0.0,1.0
25375,Michael Walker,WR,2019,14,1.90,0.0,0.0,1.0
26241,Travis Homer,RB,2019,13,2.90,0.0,1.0,0.0
29107,Nick Westbrook-Ikhine,WR,2020,11,1.70,0.0,0.0,1.0
34469,Kene Nwangwu,RB,2021,9,6.90,0.0,1.0,0.0
35419,Jacob Harris,WR,2022,12,1.60,0.0,0.0,1.0


**What we see:**
- The reported PPR points match the box score on every row. The only data errors are 12 games with a carry or target but 0 offensive snaps (a bad snap record), and we drop them.
- The independent play-by-play source agrees within 1 point on 99.85% of 36,876 games. Only 3 differ by more than 5 points (largest: Jameson Williams, 2024 week 17, 10.1 points). We keep the official box score as the target.
- Isolation Forest flags 470 games (1%), and none breaks a rule. They are real extremes, like Tyreek Hill's 57.9 points (2020 week 12) and very poor QB games, so we keep them.

### 3.5 Player pool: who we model each week
A player is in the week-*t* pool if, over their last 3 games played **before** week *t* (crossing seasons if needed),
they averaged at least 40% of their team's offensive snaps, and their team plays in week *t*. This uses only past games.
Pool players who then did not play (inactive or ruled out) are dropped; inactive lists are public before kickoff.

In [8]:
def add_pool_snap_share(candidates, games):
    """Mean snap share over each player's last N games strictly before the candidate week."""
    hist = games.sort_values(["season", "week"]).copy()
    hist["recent_snap_share"] = (hist.groupby("gsis_id")["offense_pct"]
                                 .transform(lambda s: s.fillna(0).rolling(POOL_LOOKBACK_GAMES, min_periods=1).mean()))
    hist["order"] = hist["season"] * 100 + hist["week"]
    candidates = candidates.assign(order=candidates["season"] * 100 + candidates["week"]).sort_values("order")
    return pd.merge_asof(candidates, hist[["gsis_id", "order", "recent_snap_share"]].sort_values("order"),
                         on="order", by="gsis_id", allow_exact_matches=False).drop(columns="order")


def build_candidates(rosters, games):
    """Every QB/RB/WR/TE on a roster (or who played) in a target-season week."""
    key = ["gsis_id", "season", "week"]
    r = regular_season(rosters, "game_type")
    r = r[r["position"].isin(POSITIONS)].dropna(subset=["gsis_id"]).assign(team=lambda d: fix_team(d["team"]))
    cand = pd.concat([r[key + ["team", "position", "status"]], games[key + ["team", "position"]]]).drop_duplicates(key)
    return cand[cand["season"] >= TRAIN_SEASONS[0]]


candidates = add_pool_snap_share(build_candidates(rosters, player_games), player_games)
team_has_game = candidates.set_index(["season", "week", "team"]).index.isin(team_games.set_index(["season", "week", "team"]).index)
played_keys = player_games.set_index(["gsis_id", "season", "week"]).index
candidates = candidates.assign(
    team_has_game=team_has_game,
    has_history=candidates["recent_snap_share"].notna(),
    in_pool=team_has_game & candidates["recent_snap_share"].ge(POOL_MIN_SNAP_SHARE),
    played=candidates.set_index(["gsis_id", "season", "week"]).index.isin(played_keys),
)
STATUS_GROUPS = {"ACT": "active", "INA": "game-day inactive", "RES": "injured reserve"}
pool = candidates[candidates["in_pool"]]
reason = pool["status"].map(STATUS_GROUPS).fillna("other (practice squad, cut, suspended)")
display(pd.crosstab(reason, pool["played"], margins=True).rename(columns={True: "played", False: "did not play"}))
display(pool.groupby("position")["played"].agg(pool_rows="size", played_share="mean").round(3))

played,did not play,played,All
status,,,
active,1311,22145,23456
game-day inactive,2446,0,2446
injured reserve,2815,0,2815
"other (practice squad, cut, suspended)",2007,0,2007
All,8579,22145,30724


,pool_rows,played_share
position,,
QB,6470,0.527
RB,4922,0.802
TE,6210,0.812
WR,13122,0.743


**What we see:** 30,724 pool player-weeks, and 22,145 (72%) played.
- Every pool player who was game-day inactive (2,446), on injured reserve (2,815), or on the practice squad or cut (2,007) did not play, so roster status lines up with playing.
- Of the 1,311 active players who did not play, 1,209 are QBs: benched starters who are still on the roster. That is why only 53% of the QB pool played, versus 74-81% for the other positions.

### 3.6 Expert rankings (ECR): match each snapshot to a week
FantasyPros posts weekly PPR rankings on Fridays. A snapshot belongs to the first week whose last game is on or after the
snapshot date; if a week has several snapshots we keep the latest. Players whose game kicked off on or before the snapshot date
(Thursday games, international games) are marked `played_before_snapshot` and left out of the expert comparison.
ECR is saved in its own file so it can never become a model feature.

In [9]:
def week_windows(schedules):
    s = regular_season(schedules, "game_type").assign(gameday=lambda d: pd.to_datetime(d["gameday"]))
    return s.groupby(["season", "week"])["gameday"].agg(first_game="min", last_game="max").reset_index()


def build_ecr(ecr, windows):
    e = ecr[ecr["page_type"].isin(ECR_PAGES) & (ecr["ecr_type"] == "wp")].copy()
    e["position"] = e["page_type"].map(ECR_PAGES)
    e["snapshot_date"] = pd.to_datetime(e["scrape_date"])
    e["gsis_id"] = e["id"].astype("string").map(fp_to_gsis)
    # Snapshot -> first week whose last game is on or after the snapshot date.
    snaps_dates = pd.DataFrame({"snapshot_date": e["snapshot_date"].drop_duplicates().sort_values()})
    wk = pd.merge_asof(snaps_dates, windows.sort_values("last_game"), left_on="snapshot_date",
                       right_on="last_game", direction="forward")
    wk = wk[wk["snapshot_date"] >= wk["first_game"] - pd.Timedelta(days=7)]   # drop preseason/offseason snapshots
    e = e.merge(wk[["snapshot_date", "season", "week"]].astype({"season": int, "week": int}), on="snapshot_date")
    latest = e.groupby(["season", "week", "position"])["snapshot_date"].transform("max")
    e = e[e["snapshot_date"] == latest]
    e["ecr_pos_rank"] = e.groupby(["season", "week", "position"])["ecr"].rank(method="first")
    return e[["gsis_id", "season", "week", "position", "player", "ecr", "ecr_pos_rank", "sd", "snapshot_date"]]


ecr_all = build_ecr(raw["ecr"], week_windows(raw["schedules"]))
ecr_all = ecr_all[ecr_all["season"].isin(TRAIN_SEASONS + [TEST_SEASON])]
print(f"ECR rows: {len(ecr_all):,};  mapped to a gsis_id: {ecr_all['gsis_id'].notna().mean():.1%}")
display(ecr_all.groupby("season")["week"].agg(weeks_covered="nunique", first_week="min", last_week="max"))

ECR rows: 38,690;  mapped to a gsis_id: 99.7%


,weeks_covered,first_week,last_week
season,,,
2020,11,6,16
2021,17,1,17
2022,16,2,17
2023,16,2,17
2024,14,4,17
2025,16,2,17


**What we see:**
- 99.7% of ECR rows map to a gsis_id.
- The archive covers 11 to 17 weeks a season and never week 18. 2020 starts at week 6 and 2024 at week 4, so the expert comparison only uses weeks with a snapshot.
- Training uses every week.

### 3.7 Final tables, rows kept at each step, and join rates

In [10]:
def build_player_weeks(games, cands):
    pool = cands.loc[cands["in_pool"], ["gsis_id", "season", "week", "recent_snap_share"]]
    weeks = games.merge(pool, on=["gsis_id", "season", "week"], how="inner")
    return weeks[~weeks["data_error"]].reset_index(drop=True)


def attach_ecr_flags(ecr, weeks):
    """Keep ECR rows for pool players and mark games that kicked off before the snapshot."""
    e = ecr.dropna(subset=["gsis_id"]).drop_duplicates(["gsis_id", "season", "week"])
    e = e.merge(weeks[["gsis_id", "season", "week", "kickoff"]], on=["gsis_id", "season", "week"], how="inner")
    e["played_before_snapshot"] = e["kickoff"].dt.normalize() <= e["snapshot_date"]
    return e.drop(columns="kickoff")


player_weeks = build_player_weeks(player_games, candidates)
ecr_weekly = attach_ecr_flags(ecr_all, player_weeks)

c = candidates
steps = pd.Series({
    "QB/RB/WR/TE on a roster or played, 2020-2025 regular season": len(c),
    "team has a game that week (drop byes)": c["team_has_game"].sum(),
    "has at least one earlier game": (c["team_has_game"] & c["has_history"]).sum(),
    f"recent snap share >= {POOL_MIN_SNAP_SHARE:.0%} (player pool)": c["in_pool"].sum(),
    "played (drop inactive / ruled out)": (c["in_pool"] & c["played"]).sum(),
    "not a data error (final)": len(player_weeks),
}, name="rows")
display(steps.to_frame())

ecr_weeks = player_weeks.merge(ecr_all[["season", "week"]].drop_duplicates(), on=["season", "week"])
ecr_ranked = ecr_all.dropna(subset=["gsis_id"])
top_ranked = ecr_ranked[ecr_ranked["ecr_pos_rank"] <= ecr_ranked["position"].map({"QB": 12, "RB": 24, "WR": 36, "TE": 12})]
join_rates = pd.Series({
    "snap counts": player_weeks["offense_snaps"].notna().mean(),
    "box-score stats": player_weeks["player_name"].notna().mean(),
    "schedule and Vegas lines": player_weeks["total_line"].notna().mean(),
    "ffopportunity (expected points)": player_weeks["has_expected_points"].mean(),
    "ECR: ECR rows mapped to a gsis_id": ecr_all["gsis_id"].notna().mean(),
    "ECR: pool rows (in ECR weeks) with an ECR rank": ecr_weeks.set_index(["gsis_id", "season", "week"]).index.isin(
        ecr_weekly.set_index(["gsis_id", "season", "week"]).index).mean(),
    "ECR: startable ECR players (QB12/RB24/WR36/TE12) in our pool": top_ranked.set_index(["gsis_id", "season", "week"]).index.isin(
        player_weeks.set_index(["gsis_id", "season", "week"]).index).mean(),
}, name="join rate")
display(join_rates.map("{:.1%}".format).to_frame())

,rows
"QB/RB/WR/TE on a roster or played, 2020-2025 regular season",81291
team has a game that week (drop byes),81146
has at least one earlier game,68002
recent snap share >= 40% (player pool),30724
played (drop inactive / ruled out),22145
not a data error (final),22145


,join rate
snap counts,100.0%
box-score stats,95.5%
schedule and Vegas lines,100.0%
ffopportunity (expected points),94.1%
ECR: ECR rows mapped to a gsis_id,99.7%
ECR: pool rows (in ECR weeks) with an ECR rank,98.6%
ECR: startable ECR players (QB12/RB24/WR36/TE12) in our pool,95.9%


**What we see:**
- The final table has 22,145 player-weeks (about 3,700 per season). Snap counts, schedule, and Vegas lines join for 100% of them, and expected points for 94.1%.
- 98.6% of pool players in ECR weeks have an expert rank.
- 95.9% of startable ECR players (QB12, RB24, WR36, TE12) are in our pool. The rest are 178 who did not play, 124 under the 40% snap-share bar (mostly backup RBs stepping in for an injured starter, e.g. Alexander Mattison ranked RB4 in 2020 week 6), 8 with a team mismatch, and 3 with no earlier game.
- 7.6% of ECR rows are players whose game kicked off before the snapshot (mostly Thursday games). They are kept for training but excluded from the expert comparison.
- The 12 data errors were all outside the pool, so the last step drops nothing.

In [11]:
KEY = ["gsis_id", "season", "week"]
assert not player_weeks.duplicated(KEY).any(), "duplicate player-week keys"
assert player_weeks[TARGET].notna().all(), "missing targets"
assert not ecr_weekly.duplicated(KEY).any(), "duplicate ECR keys"
assert not any(c.startswith("ecr") for c in player_weeks.columns), "ECR leaked into the modeling table"
assert player_weeks["season"].min() == TRAIN_SEASONS[0]

player_games.to_parquet(PROCESSED_DIR / "player_games.parquet", index=False)
player_weeks.to_parquet(PROCESSED_DIR / "player_weeks.parquet", index=False)
ecr_weekly.to_parquet(PROCESSED_DIR / "ecr_weekly.parquet", index=False)
print(f"Saved player_games {player_games.shape}, player_weeks {player_weeks.shape}, ecr_weekly {ecr_weekly.shape}")

Saved player_games (46945, 163), player_weeks (22145, 164), ecr_weekly (18257, 10)


## 4. Explore

## 5. Features

Goal: pre-game features for every player-week, grouped into families. Every rolling feature for week *t* uses only
games before week *t*. Steps:
1. Per-game stats to roll (points, usage shares, luck).
2. Rolling windows: last game, last 3, recency-weighted average, season to date.
3. Game script (Vegas), opponent strength, and the player's current situation (team change, injuries around them).
4. Shared evaluation code (CV folds and metrics) that every modeling section uses.
5. Pick the recency half-life, then measure what each feature family adds.

### 5.1 Per-game stats
These are the raw numbers from each played game that the rolling windows summarize. Shares are the player's part of
the team total in that game. Luck is points minus expected points: positive means the player scored more than
their opportunities usually produce.

In [12]:
def add_game_stats(games):
    g = games.sort_values(["gsis_id", "season", "week"]).reset_index(drop=True)
    g["points"] = g[TARGET]
    g["snap_share"] = g["offense_pct"].fillna(0)
    g["target_share"] = g["target_share"].fillna(0)
    g["air_yards_share"] = g["air_yards_share"].fillna(0)
    g["opportunities"] = g["attempts"].fillna(0) + g["carries"].fillna(0) + g["targets"].fillna(0)
    team_carries = g.groupby(["season", "week", "team"])["carries"].transform("sum")
    g["carry_share"] = (g["carries"].fillna(0) / team_carries.replace(0, np.nan)).fillna(0)
    g["points_over_expected"] = g["points"] - g["expected_points"]
    return g


games = add_game_stats(pd.read_parquet(PROCESSED_DIR / "player_games.parquet"))
player_weeks = pd.read_parquet(PROCESSED_DIR / "player_weeks.parquet")
injuries = load_cached("injuries", lambda: nfl.load_injuries(ALL_SEASONS))

PRODUCTION_STATS = ["points", "expected_points"]
USAGE_STATS = ["snap_share", "target_share", "carry_share", "air_yards_share", "opportunities"]
games[["gsis_id", "season", "week", "team"] + PRODUCTION_STATS + USAGE_STATS + ["points_over_expected"]].head()

,gsis_id,season,week,team,points,expected_points,snap_share,target_share,carry_share,air_yards_share,opportunities,points_over_expected
0,00-0019596,2019,1,NE,25.64,17.84,1.00,0.0,0.000000,0.0,36.0,7.80
1,00-0019596,2019,2,NE,24.66,22.49,1.00,0.0,0.057143,0.0,30.0,2.17
2,00-0019596,2019,3,NE,20.14,20.87,0.95,0.0,0.074074,0.0,44.0,-0.73
3,00-0019596,2019,4,NE,3.70,13.97,1.00,0.0,0.130435,0.0,42.0,-10.27
4,00-0019596,2019,5,NE,23.92,23.87,0.92,0.0,0.000000,0.0,42.0,0.05


**What we see:** one row per played game (46,945 from 2019 to 2025) with the stats we roll forward. Tom Brady's 2019 week 4 shows why luck matters: 3.7 points on 13.97 expected points (-10.27 luck).

### 5.2 Rolling windows over past games
For each stat we keep four views of the past, each from games before this one only:
- `last1`: the last game played
- `last3`: the mean of the last 3 games
- `recent`: a recency-weighted mean of all past games, where a game's weight halves every `half_life` games
  (with `half_life=None` it is a plain equal-weight mean of the last 8 games, the version recency has to beat)
- `season`: the mean of earlier games this season (in week 1 it falls back to `recent`)

The `recent` window runs across seasons, so early-season weeks still have history.

In [13]:
def weighted_mean(past_values, half_life):
    if half_life is None:
        return past_values.rolling(PLAIN_WINDOW_GAMES, min_periods=1).mean()
    return past_values.ewm(halflife=half_life, min_periods=1).mean()


def past_windows(g, stat, half_life):
    """Four summaries of a player's games before each row."""
    past = g.groupby("gsis_id")[stat].shift(1)          # shift(1): this game is never included
    by_player = past.groupby(g["gsis_id"])
    out = pd.DataFrame(index=g.index)
    out[f"{stat}_last1"] = past
    out[f"{stat}_last3"] = by_player.transform(lambda s: s.rolling(3, min_periods=1).mean())
    out[f"{stat}_recent"] = by_player.transform(lambda s: weighted_mean(s, half_life))
    season_past = g.groupby(["gsis_id", "season"])[stat].transform(lambda s: s.shift(1).expanding().mean())
    out[f"{stat}_season"] = season_past.fillna(out[f"{stat}_recent"])
    return out


def luck_windows(g):
    """Luck evens out, so it gets a long memory and a season-to-date mean."""
    past = g.groupby("gsis_id")["points_over_expected"].shift(1)
    return pd.DataFrame({
        "luck_recent": past.groupby(g["gsis_id"]).transform(
            lambda s: s.ewm(halflife=LUCK_HALF_LIFE_GAMES, min_periods=1).mean()),
        "luck_season": g.groupby(["gsis_id", "season"])["points_over_expected"]
                        .transform(lambda s: s.shift(1).expanding().mean()).fillna(0),
    }, index=g.index)

### 5.3 Game script and opponent strength
- **Game script** comes from the Vegas lines for this game: the team's implied points
  (half the total plus half the spread), how much the team is favored by, the total, and home or away.
- **Opponent strength** is the PPR points the opponent allowed to this position in its earlier games,
  recency-weighted the same way as the player's stats.

In [14]:
def game_script_features(g):
    return pd.DataFrame({
        "implied_team_points": g["total_line"] / 2 + g["team_favored_by"] / 2,
        "favored_by": g["team_favored_by"],
        "game_total": g["total_line"],
        "is_home": g["is_home"],
    }, index=g.index)


def opponent_features(g, half_life):
    allowed = (g.groupby(["opponent", "position", "season", "week"])["points"].sum()
               .reset_index().sort_values(["season", "week"]))
    past = allowed.groupby(["opponent", "position"])["points"].shift(1)
    allowed["opp_points_allowed"] = past.groupby([allowed["opponent"], allowed["position"]]).transform(
        lambda s: weighted_mean(s, half_life))
    keys = ["opponent", "position", "season", "week"]
    merged = g[keys].merge(allowed[keys + ["opp_points_allowed"]], on=keys, how="left")
    return merged[["opp_points_allowed"]].set_index(g.index)

### 5.4 Current situation
Features that say whether the past still describes the present:
- `games_this_season`, `weeks_since_last_game` (large after an injury), `changed_team` since the last game
- `last_season_ppg`, with `no_last_season` for players without one
- from the pre-game injury report: the player is `questionable`, the team's starting QB is out or doubtful,
  and the target and carry share that teammates listed out or doubtful usually take (`vacated_*_share`)

In [15]:
def history_features(g):
    order = g[["season", "week"]].drop_duplicates().sort_values(["season", "week"])
    week_index = order.assign(week_index=range(len(order)))
    idx = g[["season", "week"]].merge(week_index, on=["season", "week"], how="left")["week_index"].to_numpy()
    prev_team = g.groupby("gsis_id")["team"].shift(1)
    ppg = g.groupby(["gsis_id", "season"])["points"].mean().rename("last_season_ppg").reset_index()
    ppg["season"] += 1
    last_season = g[["gsis_id", "season"]].merge(ppg, on=["gsis_id", "season"], how="left")["last_season_ppg"]
    return pd.DataFrame({
        "games_this_season": g.groupby(["gsis_id", "season"]).cumcount(),
        "weeks_since_last_game": idx - pd.Series(idx, index=g.index).groupby(g["gsis_id"]).shift(1),
        "changed_team": (prev_team.notna() & (prev_team != g["team"])).astype(int),
        "last_season_ppg": last_season.fillna(0).to_numpy(),
        "no_last_season": last_season.isna().astype(int).to_numpy(),
    }, index=g.index)


def clean_injury_report(injuries):
    inj = regular_season(injuries, "game_type").dropna(subset=["gsis_id"]).copy()
    inj = inj.astype({"season": int, "week": int}).assign(team=lambda d: fix_team(d["team"]))
    inj = inj.sort_values("date_modified").drop_duplicates(["gsis_id", "season", "week"], keep="last")
    inj["out_or_doubtful"] = inj["report_status"].isin(["Out", "Doubtful"])
    return inj[["gsis_id", "season", "week", "team", "report_status", "out_or_doubtful"]]


def usage_before_week(g, keys, half_life):
    """Each player's recency-weighted usage through their last game before the given (season, week)."""
    hist = g[["gsis_id", "season", "week", "position", "snap_share", "target_share", "carry_share"]].copy()
    for stat in ["snap_share", "target_share", "carry_share"]:
        hist[stat] = hist.groupby("gsis_id")[stat].transform(lambda s: weighted_mean(s, half_life))
    hist["order"] = (hist["season"] * 100 + hist["week"]).astype("int64")
    keys = keys.assign(order=(keys["season"] * 100 + keys["week"]).astype("int64")).sort_values("order")
    return pd.merge_asof(keys, hist.drop(columns=["season", "week"]).sort_values("order"),
                         on="order", by="gsis_id", allow_exact_matches=False)


def injury_features(g, injury_report, half_life):
    key = ["gsis_id", "season", "week"]
    listed = usage_before_week(g, injury_report[injury_report["out_or_doubtful"]], half_life)
    team_week = ["season", "week", "team"]
    vacated = listed.groupby(team_week)[["target_share", "carry_share"]].sum().add_prefix("vacated_")
    qb_out = (listed[(listed["position"] == "QB") & (listed["snap_share"] >= 0.5)]
              .groupby(team_week).size().gt(0).rename("starting_qb_out"))
    out = g[key + ["team"]].merge(vacated.reset_index(), on=team_week, how="left")
    out = out.merge(qb_out.reset_index(), on=team_week, how="left")
    status = injury_report[key + ["report_status"]]
    out = out.merge(status, on=key, how="left")
    return pd.DataFrame({
        "questionable": out["report_status"].eq("Questionable").astype(int).to_numpy(),
        "starting_qb_out": out["starting_qb_out"].fillna(False).astype(int).to_numpy(),
        "vacated_target_share": out["vacated_target_share"].fillna(0).to_numpy(),
        "vacated_carry_share": out["vacated_carry_share"].fillna(0).to_numpy(),
    }, index=g.index)


injury_report = clean_injury_report(injuries)
print(f"Injury report rows: {len(injury_report):,};  listed out or doubtful: {injury_report['out_or_doubtful'].mean():.1%}")

Injury report rows: 38,599;  listed out or doubtful: 21.6%


**What we see:** 38,599 injury-report rows from 2019 to 2025, 21.6% of them listed out or doubtful. The report is published before each game, so it is pre-game information.

### 5.5 Build the feature table
All families are computed on every played game (so the history is complete), then cut down to the modeling rows.

In [16]:
def build_features(games, half_life):
    g = games
    parts = [past_windows(g, stat, half_life) for stat in PRODUCTION_STATS + USAGE_STATS]
    parts += [luck_windows(g), game_script_features(g), opponent_features(g, half_life),
              history_features(g), injury_features(g, injury_report, half_life),
              pd.get_dummies(g["position"], prefix="pos", dtype=int)]
    table = pd.concat([g[["gsis_id", "season", "week", "position", "team", "kickoff", TARGET]]] + parts, axis=1)
    return table.merge(player_weeks[["gsis_id", "season", "week"]], on=["gsis_id", "season", "week"])


def window_columns(stats):
    return [f"{s}_{w}" for s in stats for w in ["last1", "last3", "recent", "season"]]


FEATURE_FAMILIES = {
    "position": [f"pos_{p}" for p in POSITIONS],
    "recent_production": window_columns(PRODUCTION_STATS),
    "usage": window_columns(USAGE_STATS),
    "luck": ["luck_recent", "luck_season"],
    "game_script": ["implied_team_points", "favored_by", "game_total", "is_home"],
    "opponent": ["opp_points_allowed"],
    "situation": ["games_this_season", "weeks_since_last_game", "changed_team", "last_season_ppg", "no_last_season",
                  "questionable", "starting_qb_out", "vacated_target_share", "vacated_carry_share"],
}
ALL_FEATURES = [c for cols in FEATURE_FAMILIES.values() for c in cols]

features_by_half_life = {hl: build_features(games, hl) for hl in HALF_LIFE_CANDIDATES + [None]}
example = features_by_half_life[HALF_LIFE_CANDIDATES[0]]
print(f"{len(example):,} rows x {len(ALL_FEATURES)} features in {len(FEATURE_FAMILIES)} families")
assert example[ALL_FEATURES].notna().all().all(), "missing feature values"

22,145 rows x 48 features in 7 families


**What we see:** 22,145 rows and 48 features in 7 families, with no missing values.
- Every pool player has at least one earlier game, so the windows always have history.
- Players with no prior season get `last_season_ppg = 0` and `no_last_season = 1` (12.0% of training rows).

**Do the situation features point the right way?** Mean PPR points in training seasons, with and without each flag.

In [17]:
train_example = example[example["season"].isin(TRAIN_SEASONS)]
display(pd.concat({flag: train_example.groupby(["position", flag])[TARGET].mean().unstack()
                   for flag in ["changed_team", "questionable", "starting_qb_out"]}, axis=1).round(2))

changed_team       questionable        starting_qb_out       
changed_team            0     1            0      1               0      1
position                                                                  
QB                  15.45  7.47        15.24  13.00           15.40  10.99
RB                  12.44  7.15        12.38  11.74           12.43  10.76
TE                   6.51  3.96         6.49   5.83            6.50   5.91
WR                   9.85  5.15         9.79   9.16            9.86   8.05

**What we see:** every flag lowers scoring, as expected.
- A player in their first game for a new team scores far less (WR 9.85 vs. 5.15 points), so old usage overstates them.
- A WR whose starting QB is out drops from 9.86 to 8.05, and a questionable tag costs about half a point to 2 points.
- Changed team (2.2% of rows), questionable (5.6%), and QB out (5.9%) are rare, so they matter for a few players a week, not the average.

### 5.6 Shared evaluation: time-ordered CV and metrics
Every modeling section uses these two functions, so all results are comparable.
- `cv_folds`: expanding window. To validate on season *s*, train on every season from 2020 to *s*-1. 2025 is never used.
- `evaluate`: point error (RMSE, MAE) and ranking quality within each position-week: Spearman rank correlation and the
  share of our top 12 who actually finished top 12.

In [18]:
def cv_folds(data):
    for season in CV_VALID_SEASONS:
        train = data.index[data["season"].between(TRAIN_SEASONS[0], season - 1)]
        valid = data.index[data["season"] == season]
        yield season, train, valid


def top_n_hit_rate(week):
    predicted_top = set(week.nlargest(TOP_N, "pred").index)
    actual_top = set(week.nlargest(TOP_N, TARGET).index)
    return len(predicted_top & actual_top) / TOP_N


def evaluate(data, pred):
    d = data.assign(pred=np.asarray(pred))
    groups = d.groupby(["season", "week", "position"])
    return {
        "rmse": float(np.sqrt(((d["pred"] - d[TARGET]) ** 2).mean())),
        "mae": float((d["pred"] - d[TARGET]).abs().mean()),
        "spearman": float(groups.apply(lambda w: w["pred"].corr(w[TARGET], method="spearman")).mean()),
        "top12_hit_rate": float(groups.apply(top_n_hit_rate).mean()),
    }


def cross_validate(model, data, columns):
    """Fit on each expanding-window fold; return mean metrics and the out-of-fold predictions."""
    from sklearn.base import clone
    preds = pd.Series(np.nan, index=data.index)
    for season, train, valid in cv_folds(data):
        fitted = clone(model).fit(data.loc[train, columns], data.loc[train, TARGET])
        preds.loc[valid] = fitted.predict(data.loc[valid, columns])
    scored = preds.notna()
    return evaluate(data[scored], preds[scored]), preds

### 5.7 How much should recent games count?
The same Ridge model, using production and usage features, built four ways: equal weight over the last 8 games, and
recency-weighted with a half-life of 3, 5, or 8 games. We keep the half-life with the best CV rank correlation.

In [19]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
recency_columns = FEATURE_FAMILIES["position"] + FEATURE_FAMILIES["recent_production"] + FEATURE_FAMILIES["usage"]


def label(half_life):
    return f"equal weight, last {PLAIN_WINDOW_GAMES}" if half_life is None else f"half-life {half_life} games"


def train_rows(table):
    return table[table["season"].isin(TRAIN_SEASONS)].reset_index(drop=True)


recency_results = pd.DataFrame({
    label(hl): cross_validate(ridge, train_rows(table), recency_columns)[0]
    for hl, table in features_by_half_life.items()
}).T.round(4)
display(recency_results)

HALF_LIFE = max(HALF_LIFE_CANDIDATES, key=lambda hl: recency_results.loc[label(hl), "spearman"])
features = features_by_half_life[HALF_LIFE]
print(f"Chosen half-life: {HALF_LIFE} games")

,rmse,mae,spearman,top12_hit_rate
half-life 3 games,6.8969,5.2975,0.4959,0.4784
half-life 5 games,6.8791,5.2846,0.4999,0.4819
half-life 8 games,6.8725,5.2824,0.5014,0.4826
half-life 12 games,6.8730,5.2867,0.5013,0.4857
half-life 16 games,6.8753,5.2909,0.5012,0.4861
"equal weight, last 8",6.9207,5.3153,0.4901,0.4765


Chosen half-life: 8 games


**What we see:**
- Recency weighting beats equal weighting at every half-life. The equal-weight last-8 mean has the worst rank correlation (0.490) and RMSE (6.92).
- A half-life of 8 games is best (Spearman 0.501, RMSE 6.87), and 12 and 16 are within 0.0002 of it.
- So recent games should count more, but the weight should fade slowly: a game 8 games back still counts half as much as last week's.

### 5.8 What each feature family adds
Families are added one at a time to Ridge and a random forest (both untuned here; tuning is in section 6).
Two no-model baselines show the floor: last game's points and the mean of the last 3 games.

In [20]:
from sklearn.ensemble import RandomForestRegressor

train_features = train_rows(features)
forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=20, max_features=0.5, n_jobs=-1, random_state=SEED)


def baseline_scores(data):
    scored = data[data["season"].isin(CV_VALID_SEASONS)]
    return {"baseline: last game": evaluate(scored, scored["points_last1"]),
            "baseline: last 3 games": evaluate(scored, scored["points_last3"])}


def family_ladder(data, model):
    results, columns = {}, []
    for family, cols in FEATURE_FAMILIES.items():
        columns = columns + cols
        if family != "position":
            results[f"+ {family}"] = cross_validate(model, data, columns)[0]
    return results


family_results = pd.concat({
    "no model": pd.DataFrame(baseline_scores(train_features)).T,
    "Ridge": pd.DataFrame(family_ladder(train_features, ridge)).T,
    "Random forest": pd.DataFrame(family_ladder(train_features, forest)).T,
}).round(4)
display(family_results)

rmse     mae  spearman  top12_hit_rate
no model      baseline: last game     9.0658  6.7694    0.3532          0.4059
              baseline: last 3 games  7.5943  5.7843    0.4234          0.4456
Ridge         + recent_production     6.9118  5.3199    0.4917          0.4792
              + usage                 6.8725  5.2824    0.5014          0.4826
              + luck                  6.8715  5.2828    0.5016          0.4826
              + game_script           6.8490  5.2345    0.5071          0.4915
              + opponent              6.8464  5.2205    0.5086          0.4938
              + situation             6.8090  5.1939    0.5146          0.4896
Random forest + recent_production     6.9364  5.3250    0.4855          0.4792
              + usage                 6.8929  5.3067    0.4956          0.4823
              + luck                  6.8888  5.3093    0.4964          0.4807
              + game_script           6.8415  5.2598    0.5081          0.4896
              + opponent              6.8374  5.2563    0.5079          0.4873
              + situation             6.8092  5.2360    0.5140          0.4888

**What we see:**
- Both models beat the no-model baselines by a wide margin: RMSE 6.81 vs. 7.59 for the last-3 average, and rank correlation 0.515 vs. 0.423.
- **Biggest gains:** usage (+0.010 Spearman for Ridge), the situation family (+0.006 Spearman, -0.037 RMSE, the largest RMSE drop after the first family), and game script (+0.006).
- **Luck adds almost nothing** (+0.0002). Expected points already carries most of that signal. We keep it for now and let the regularized models in section 6 decide.
- Ridge and the untuned random forest end level (Spearman 0.515 vs. 0.514), so model choice is open for section 6.

### 5.9 Leakage checks and a spot check
- **Leakage test:** for 300 random rows, recompute three features by hand from only the games before that week and
  compare to the table.
- **ECR check:** no ECR column is in the features.
- **Spot check:** Christian McCaffrey, traded from CAR to SF in 2022 week 7, and back from injured reserve in 2024 week 10.

In [21]:
def recompute_by_hand(row):
    past = games[(games["gsis_id"] == row["gsis_id"]) &
                 ((games["season"] < row["season"]) | ((games["season"] == row["season"]) & (games["week"] < row["week"])))]
    return pd.Series({"points_last1": past["points"].iloc[-1],
                      "points_last3": past["points"].tail(3).mean(),
                      "games_this_season": (past["season"] == row["season"]).sum()})


sample = features.sample(300, random_state=SEED)
by_hand = sample.apply(recompute_by_hand, axis=1)
mismatches = (by_hand - sample[by_hand.columns]).abs().gt(1e-9).sum()
print("Leakage test mismatches per feature:", mismatches.to_dict())
assert mismatches.sum() == 0
assert not [c for c in ALL_FEATURES if "ecr" in c.lower()], "ECR column in features"
assert not features.duplicated(["gsis_id", "season", "week"]).any()

cmc = features[(features["gsis_id"] == "00-0033280") &
               (((features["season"] == 2022) & features["week"].between(5, 9)) |
                ((features["season"] == 2024) & features["week"].between(10, 12)))]
display(cmc[["season", "week", "team", "changed_team", "weeks_since_last_game", "games_this_season",
             "points_last1", "points_recent", "snap_share_last3", TARGET]])

Leakage test mismatches per feature: {'points_last1': 0, 'points_last3': 0, 'games_this_season': 0}


,season,week,team,changed_team,weeks_since_last_game,games_this_season,points_last1,points_recent,snap_share_last3,fantasy_points_ppr
5529,2022,5,CAR,0,1.0,4.0,25.8,21.790617,0.860000,23.40
5530,2022,6,CAR,0,1.0,5.0,23.4,21.933959,0.843333,22.80
5531,2022,7,SF,1,1.0,6.0,22.8,22.010628,0.863333,8.20
5532,2022,8,SF,0,1.0,7.0,8.2,20.794715,0.670000,40.26
5558,2024,10,SF,0,11.0,0.0,13.1,23.600816,0.770000,16.70
5559,2024,11,SF,0,1.0,1.0,16.7,23.024895,0.806667,14.60
5560,2024,12,SF,0,1.0,2.0,14.6,22.322102,0.786667,7.80


**What we see:**
- The leakage test matches on all 300 rows.
- In McCaffrey's first SF game (2022 week 7), `changed_team` = 1. The return in 2024 week 10 shows `weeks_since_last_game` = 11 and `games_this_season` = 0.
- At that point McCaffrey's recency-weighted mean (23.6) is still built on the 2023 season, which is the carry-over across seasons working as intended.

In [22]:
features.to_parquet(PROCESSED_DIR / "features.parquet", index=False)
(PROCESSED_DIR / "feature_families.json").write_text(pd.Series(FEATURE_FAMILIES).to_json(indent=2))
print(f"Saved features {features.shape} with half-life {HALF_LIFE}")

Saved features (22145, 55) with half-life 8


## 6. Points regression (main task)

### Predictions handed to later phases (keep this the last part of section 6)
Tiers (section 7), clustering (section 8), and the final test (section 9) read the regression's out-of-fold predictions
and settings from two files. Until the model comparison above picks a winner, this saves the untuned Ridge baseline from
section 5.8. To publish the chosen model, change only `REGRESSION_CHOICE` and `make_regression_model()`; section 9 reuses both.
- `preds_regression.parquet`: one prediction per player-week in the CV seasons (2022-2024), made by a model that never saw that season
- `regression_choice.json`: the model and settings, so section 9 can rebuild the same model for 2025

In [23]:
import json

REGRESSION_CHOICE = {
    "status": "baseline",                       # becomes "final" once section 6 picks a model
    "model": "Ridge",
    "params": {"alpha": 1.0},
    "features": ALL_FEATURES,
    "training_window": "all seasons before the validation season",
    "sample_weight_half_life_seasons": None,    # None = every season counts the same
    "refit": "once per validation season",
}


def make_regression_model(choice):
    return make_pipeline(StandardScaler(), Ridge(**choice["params"]))


def save_regression_outputs(data, choice):
    scores, oof = cross_validate(make_regression_model(choice), data, choice["features"])
    scored = oof.notna()
    preds = data.loc[scored, KEY + ["position"]].assign(pred=oof[scored], model=choice["model"])
    preds.to_parquet(PROCESSED_DIR / "preds_regression.parquet", index=False)
    (PROCESSED_DIR / "regression_choice.json").write_text(json.dumps(choice, indent=2))
    return preds, scores


regression_preds, regression_scores = save_regression_outputs(train_features, REGRESSION_CHOICE)
print(f"Saved {len(regression_preds):,} out-of-fold predictions "
      f"({regression_preds['season'].min()}-{regression_preds['season'].max()}), status: {REGRESSION_CHOICE['status']}")
pd.Series(regression_scores).round(4).to_frame("CV score")

Saved 11,207 out-of-fold predictions (2022-2024), status: baseline


,CV score
rmse,6.8090
mae,5.1939
spearman,0.5146
top12_hit_rate,0.4896


**What we see:**
- 11,207 out-of-fold predictions for 2022-2024, with the same scores as the full-feature Ridge in section 5.8 (RMSE 6.81, Spearman 0.515, top-12 hit rate 0.490).
- This is the placeholder later phases build on. When section 6 picks a model, its predictions replace this file with the same columns.

## 7. Tiers (extension)

## 8. Usage clustering (extension)

## 9. Final test vs. experts (2025)

## 10. Analysis and wrap-up

### What we tried and didn't keep

| Idea | Why we dropped it |
|---|---|

### Societal impact